In [ ]:
from IPython.display import clear_output
class UltimateTicTacToe:
    def __init__(self):
        # Grille 9x9 (9 sous-grilles de 3x3)
        self.board = [['°' for _ in range(9)] for _ in range(9)]
        # État des 9 sous-grilles ('⸱' = en cours, 'X', 'O', ou 'D' pour Draw)
        self.macro_board = ['°' for _ in range(9)]
        self.current_player = 'X'
        self.next_macro = -1  # -1 signifie que le joueur peut jouer n'importe où

    def get_macro_name(self, index):
        names = [
            "en Haut à Gauche", "en Haut au Centre", "en Haut à Droite",
            "au Milieu à Gauche", "au Centre", "au Milieu à Droite",
            "en Bas à Gauche", "en Bas au Centre", "en Bas à Droite"
        ]
        return names[index] if 0 <= index < 9 else "Inconnue"

    def print_board(self):
        clear_output()
        print("\n    1 2 3   4 5 6   7 8 9")
        print("  === ULTIMATE TIC TAC TOE ===")

        print("\n  ╔═══════════════════╗")
        for i in range(9):
            if i % 3 == 0 and i != 0:
                print("  " + "╠" + "═" * 17 + "╣")
            row_str = f"{i+1} ║ "
            for j in range(9):
                if j % 3 == 0 and j != 0:
                    row_str += "║ "
                row_str += self.board[i][j] + " "
            print(row_str)
        print("  ╚═══════════════════╝\n")

        print("État des sous-grilles (Macro-board) :")
        for i in range(0, 9, 3):
            print(f" {self.macro_board[i]} | {self.macro_board[i+1]} | {self.macro_board[i+2]} ")
        print()

    def check_win(self, grid):
        win_lines = [
            [0, 1, 2], [3, 4, 5], [6, 7, 8], # Lignes
            [0, 3, 6], [1, 4, 7], [2, 5, 8], # Colonnes
            [0, 4, 8], [2, 4, 6]             # Diagonales
        ]
        for line in win_lines:
            if grid[line[0]] == grid[line[1]] == grid[line[2]] and grid[line[0]] not in ['⸱', 'D']:
                return grid[line[0]]
        if '°' not in grid:
            return 'D' # Égalité
        return '°'

    def get_macro_index(self, row, col):
        return (row // 3) * 3 + (col // 3)

    def get_local_index(self, row, col):
        return (row % 3) * 3 + (col % 3)

    def is_valid_move(self, row, col):
        if not (0 <= row < 9 and 0 <= col < 9):
            return False
        if self.board[row][col] != '⸱':
            return False

        macro_idx = self.get_macro_index(row, col)

        if self.macro_board[macro_idx] != '⸱':
            return False

        if self.next_macro != -1 and self.next_macro != macro_idx:
            return False

        return True

    def play_move(self, row, col):
        if not self.is_valid_move(row, col):
            return False

        self.board[row][col] = self.current_player
        macro_idx = self.get_macro_index(row, col)
        local_idx = self.get_local_index(row, col)

        sub_grid = []
        r_start, c_start = (macro_idx // 3) * 3, (macro_idx % 3) * 3
        for r in range(r_start, r_start + 3):
            for c in range(c_start, c_start + 3):
                sub_grid.append(self.board[r][c])

        sub_winner = self.check_win(sub_grid)
        if sub_winner != '⸱':
            self.macro_board[macro_idx] = sub_winner

        if self.macro_board[local_idx] != '⸱':
            self.next_macro = -1
        else:
            self.next_macro = local_idx

        self.current_player = 'O' if self.current_player == 'X' else 'X'
        return True

    def is_game_over(self):
        winner = self.check_win(self.macro_board)
        if winner != '⸱':
            self.print_board()
            if winner == 'D':
                print("Match Nul global !")
            else:
                print(f"🎉 Le joueur {winner} a remporté la partie globale ! 🎉")
            return True
        return False

# --- Boucle de jeu principale ---
if __name__ == "__main__":
    game = UltimateTicTacToe()

    while not game.is_game_over():
        game.print_board()

        if game.next_macro == -1:
            print(f"C'est à {game.current_player} de jouer. Vous pouvez jouer dans N'IMPORTE QUELLE sous-grille disponible.")
        else:
            macro_name = game.get_macro_name(game.next_macro)
            print(f"C'est à {game.current_player} de jouer. Vous DEVEZ jouer dans la sous-grille : {macro_name}")

        try:
            user_input = input("Entrez votre coup (Ligne Colonne de 1 à 9, ex: 4 5) : ")
            input_row, input_col = map(int, user_input.split())

            # Conversion des coordonnées 1-9 vers les index internes 0-8
            internal_row = input_row - 1
            internal_col = input_col - 1

            if not game.play_move(internal_row, internal_col):
                print("\n❌ Mouvement invalide. Vérifiez les coordonnées et les restrictions de sous-grille.")
        except (ValueError, IndexError):
            print("\n❌ Entrée invalide. Veuillez entrer deux nombres entre 1 et 9 séparés par un espace.")

In [ ]:
from IPython.display import clear_output
import time, math, copy, random

# --- CONSTANTE TEMPS MAX ---
TEMPS_MAX = 3.0  # secondes

# --- EXCEPTIONS ---
class TimeoutError(Exception):
    pass

# --- LOGIQUE DU JEU ---
class UltimateTicTacToe:
    def __init__(self):
        self.board = [['⸱' for _ in range(9)] for _ in range(9)]
        self.macro_board = ['⸱' for _ in range(9)]
        self.current_player = 'X'
        self.next_macro = -1

    def print_board(self):
        clear_output()
        print("\n    1 2 3   4 5 6   7 8 9")
        print("  ╔═════════════════════╗")
        for i in range(9):
            if i % 3 == 0 and i != 0:
                print("  ╠═══════╬═══════╬═══════╣")
            row_str = f"{i+1} ║ "
            for j in range(9):
                if j % 3 == 0 and j != 0:
                    row_str += "║ "
                row_str += self.board[i][j] + " "
            print(row_str + "║")
        print("  ╚═════════════════════╝")

    def check_win(self, grid):
        win_lines = [[0,1,2],[3,4,5],[6,7,8],[0,3,6],[1,4,7],[2,5,8],[0,4,8],[2,4,6]]
        for l in win_lines:
            if grid[l[0]] == grid[l[1]] == grid[l[2]] and grid[l[0]] not in ['⸱', 'D']:
                return grid[l[0]]
        return 'D' if '⸱' not in grid else '⸱'

    def is_valid_move(self, r, c):
        if not (0 <= r < 9 and 0 <= c < 9) or self.board[r][c] != '⸱':
            return False
        m_idx = (r // 3) * 3 + (c // 3)
        if self.macro_board[m_idx] != '⸱':
            return False
        return self.next_macro == -1 or self.next_macro == m_idx

    def play_move(self, r, c):
        if not self.is_valid_move(r, c):
            return None

        m_idx = (r // 3) * 3 + (c // 3)
        prev_macro_value = self.macro_board[m_idx]
        prev_next = self.next_macro

        self.board[r][c] = self.current_player

        sub = [self.board[ri][ci]
              for ri in range((m_idx//3)*3, (m_idx//3)*3+3)
              for ci in range((m_idx%3)*3, (m_idx%3)*3+3)]

        self.macro_board[m_idx] = self.check_win(sub)

        target_macro = (r % 3) * 3 + (c % 3)
        self.next_macro = target_macro if self.macro_board[target_macro] == '⸱' else -1

        self.current_player = 'O' if self.current_player == 'X' else 'X'

        return (r, c, m_idx, prev_macro_value, prev_next)

    def undo_move(self, move_info):
        if move_info is None:
            return

        r, c, m_idx, prev_macro_value, prev_next = move_info

        self.board[r][c] = '⸱'
        self.macro_board[m_idx] = prev_macro_value
        self.next_macro = prev_next
        self.current_player = 'O' if self.current_player == 'X' else 'X'

    def is_game_over(self):
        res = self.check_win(self.macro_board)
        if res != '⸱':
            self.print_board()
            print(f"\nFIN : {'Match nul !' if res == 'D' else 'Victoire de ' + res}")
            return True
        return False

def trouver_grille(ligne, colonne):
    return ((ligne - 1) % 3) * 3 + ((colonne - 1) % 3) + 1

# --- HEURISTIQUE ---
def evaluation(jeu):
    WIN_LINES = [[0,1,2],[3,4,5],[6,7,8],[0,3,6],[1,4,7],[2,5,8],[0,4,8],[2,4,6]]

    poids_macro = [3,2,3,
                   2,4,2,
                   3,2,3]

    res = jeu.check_win(jeu.macro_board)
    if res == 'O': return 1000
    if res == 'X': return -1000

    score = 0

    for i, m in enumerate(jeu.macro_board):
        if m == 'O': score += 10 * poids_macro[i]
        elif m == 'X': score -= 10 * poids_macro[i]

    for m_idx in range(9):
        if jeu.macro_board[m_idx] != '⸱':
            continue

        row_start = (m_idx // 3) * 3
        col_start = (m_idx % 3) * 3
        sub = [jeu.board[row_start + r][col_start + c]
               for r in range(3) for c in range(3)]

        for line in WIN_LINES:
            cells = [sub[i] for i in line]
            o_count = cells.count('O')
            x_count = cells.count('X')
            empty = cells.count('⸱')

            if o_count > 0 and x_count > 0:
                continue

            if empty == 1:
                if o_count == 2: score += 3
                elif x_count == 2: score -= 3

            elif empty == 2:
                if o_count == 1: score += 1
                elif x_count == 1: score -= 1

        for line in WIN_LINES:
            macro_cells = [jeu.macro_board[i] for i in line]
            o_count = macro_cells.count('O')
            x_count = macro_cells.count('X')
            empty = macro_cells.count('⸱')

            if o_count > 0 and x_count > 0:
                continue

            if empty == 1:
                if o_count == 2: score += 20
                elif x_count == 2: score -= 20

    if jeu.next_macro != -1:
        m = jeu.next_macro
        if jeu.macro_board[m] == '⸱':
            row_start = (m // 3) * 3
            col_start = (m % 3) * 3
            sub = [jeu.board[row_start + r][col_start + c]
                   for r in range(3) for c in range(3)]

            for line in WIN_LINES:
                cells = [sub[i] for i in line]
                o_count = cells.count('O')
                x_count = cells.count('X')
                empty = cells.count('⸱')

                if o_count == 2 and empty == 1:
                    score += 8
                elif x_count == 2 and empty == 1:
                    score -= 8

    return score

# --- MINIMAX AVEC TEMPS GLOBAL ---
def minimax(jeu, joueur, alpha, beta, prof, debut, limite):
    if time.time() - debut > limite:
        raise TimeoutError()

    if prof == 0 or jeu.check_win(jeu.macro_board) != '⸱':
        return evaluation(jeu)

    coups = [(r, c) for r in range(9) for c in range(9) if jeu.is_valid_move(r, c)]

    if joueur == "MAX":
        val = -math.inf
        for r, c in coups:
            move_info = jeu.play_move(r, c)
            val = max(val, minimax(jeu, "MIN", alpha, beta, prof-1, debut, limite))
            jeu.undo_move(move_info)

            alpha = max(alpha, val)
            if alpha >= beta:
                break
        return val

    else:
        val = math.inf
        for r, c in coups:
            move_info = jeu.play_move(r, c)
            val = min(val, minimax(jeu, "MAX", alpha, beta, prof-1, debut, limite))
            jeu.undo_move(move_info)

            beta = min(beta, val)
            if alpha >= beta:
                break
        return val

# --- JEU ---
if __name__ == "__main__":
    game = UltimateTicTacToe()
    grilleact = 0
    nbtour = 1


    print("--- CHOIX DU PREMIER JOUEUR ---")
    choix = input("1: Humain (X), 2: IA (O), 3: Aléatoire. Votre choix : ")
    if choix == '2': game.current_player = 'O'
    elif choix == '3': game.current_player = random.choice(['X', 'O'])

    while not game.is_game_over():
        game.print_board()

        if game.current_player == 'X':
            print(f"\nTour n°{nbtour} c'est à X de jouer (Humain).")
            if nbtour >1 :
                print(f"vous devez jouer dans la grille {grilleact}")
                print(f"Score : {evaluation(game)}")
                print (f"L'IA a joué en {best_c[0]+1} {best_c[1]+1} (Temps de l'IA : {temps:.3f}s)")
            try:
                txt = input("Coup (Ligne Colonne 1-9) : ")
                r, c = map(int, txt.split())
                if not game.play_move(r-1, c-1):
                    print("Coup invalide !")
                    time.sleep(1)
            except:
                print("Erreur de saisie.")
                time.sleep(1)

        else:
            print("\nL'IA réfléchit...")
            start = time.time()
            best_s = -math.inf
            best_c = None

            possibles = [(r, c) for r in range(9) for c in range(9) if game.is_valid_move(r, c)]

            try:
                for c in possibles:
                    if time.time() - start > TEMPS_MAX:
                        raise TimeoutError()

                    sim = copy.deepcopy(game)
                    sim.play_move(c[0], c[1])
                    if (nbtour>20) :
                        s = minimax(sim, "MIN", -math.inf, math.inf, 5, start, TEMPS_MAX)
                    elif (nbtour>3) :
                        s = minimax(sim, "MIN", -math.inf, math.inf, 3, start, TEMPS_MAX)
                    else :
                        s = minimax(sim, "MIN", -math.inf, math.inf, 1, start, TEMPS_MAX)


                    if s > best_s:
                        best_s, best_c = s, c

            except TimeoutError:
                pass

            if not best_c:
                best_c = possibles[0]

            temps = time.time() - start
            game.play_move(best_c[0], best_c[1])
            grilleact = trouver_grille(best_c[0]+1, best_c[1]+1)
            time.sleep(1)

        nbtour += 1


    1 2 3   4 5 6   7 8 9
  ╔═════════════════════╗
1 ║ ⸱ ⸱ ⸱ ║ ⸱ ⸱ ⸱ ║ ⸱ ⸱ ⸱ ║
2 ║ ⸱ O ⸱ ║ ⸱ ⸱ ⸱ ║ ⸱ ⸱ ⸱ ║
3 ║ ⸱ ⸱ ⸱ ║ ⸱ ⸱ ⸱ ║ ⸱ ⸱ ⸱ ║
  ╠═══════╬═══════╬═══════╣
4 ║ ⸱ ⸱ ⸱ ║ X ⸱ ⸱ ║ ⸱ ⸱ ⸱ ║
5 ║ ⸱ ⸱ ⸱ ║ ⸱ O ⸱ ║ ⸱ ⸱ ⸱ ║
6 ║ ⸱ ⸱ ⸱ ║ ⸱ ⸱ ⸱ ║ ⸱ ⸱ ⸱ ║
  ╠═══════╬═══════╬═══════╣
7 ║ ⸱ ⸱ ⸱ ║ ⸱ ⸱ ⸱ ║ ⸱ ⸱ ⸱ ║
8 ║ ⸱ ⸱ ⸱ ║ ⸱ ⸱ ⸱ ║ ⸱ ⸱ ⸱ ║
9 ║ ⸱ ⸱ ⸱ ║ ⸱ ⸱ ⸱ ║ ⸱ ⸱ ⸱ ║
  ╚═════════════════════╝

Tour n°4 c'est à X de jouer (Humain).
vous devez jouer dans la grille 5
Score : 5
L'IA a joué en 2 2 (Temps de l'IA : 0.008s)
